<a href="https://colab.research.google.com/github/MikaTan2007/bc-wildfire-prediction/blob/main/bc_wildfire_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BC Wildfire Prediction Model

## Setup

In [ ]:
!pip install geemap

In [ ]:
import ee
import geemap

ee.Authenticate()

ee.Initialize(project='bc-wildfire-491217')

### Defining British Columbia

In [ ]:
bc_bounds = ee.Geometry.BBox(-139.059399, 48.232274, -114.077738, 60.001826) #Bounds of BC
map = geemap.Map(center=[54.0, -125.0], zoom=5)

map.add_basemap('SATELLITE')
map.add_basemap('TERRAIN')

map.addLayer(bc_bounds, {'color': 'blue'}, 'BC Bounding Box') #Blue Filled Box
map.addLayer(ee.Image().paint(bc_bounds, 0, 0.5), {}, 'BC Outline Box') #Outline Box

map

Map(center=[54.0, -125.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

### ECMWF/ERA5/DAILY Dataset for July 2019

In [ ]:
era5 = (
    ee.ImageCollection("ECMWF/ERA5/DAILY")
    .select(['mean_2m_air_temperature', 'total_precipitation'])
    .filterBounds(bc_bounds)
    .filterDate('2019-07-01', '2019-07-31') # July 2019
)

# Calculate the mean weather over BC for that entire month
mean_weather = era5.mean().clip(bc_bounds)

# Calculate the single average temperature for the WHOLE province
mean_scalar = mean_weather.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=bc_bounds,
    scale=27830 # ERA5 scale is roughly 27.8km
)

# .getInfo() forces Earth Engine to compute it and return a Python dictionary
result = mean_scalar.getInfo()

# Convert Kelvin to Celsius
temp_c = result['mean_2m_air_temperature'] - 273.15
print(f"Average BC Temperature for July 2019: {temp_c:.2f}°C")

map = geemap.Map(center=[54.0, -125.0], zoom=5)
temp_viz = {
    'min': 273.15,
    'max': 303.15,
    'palette': ['blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}

map.add_basemap('SATELLITE')
map.add_basemap('TERRAIN')

map.addLayer(
    mean_weather.select('mean_2m_air_temperature'),
    temp_viz,
    'Mean Temperature July 2019'
)

map.addLayer(ee.Image().paint(bc_bounds, 0, 0.5), {}, 'BC Outline Box') #Outline Box

map

Average BC Temperature for July 2019: 13.73°C


Map(center=[54.0, -125.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

### ERA5-Land Daily Dataset July 2019

In [ ]:
# Load ERA5-Land Daily dataset
era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .select(['temperature_2m', 'total_precipitation_sum']) # Updated band names
    .filterBounds(bc_bounds)
    .filterDate('2019-07-01', '2019-07-31')
)

# Calculate the mean weather over BC for July
mean_weather = era5.mean().clip(bc_bounds)

# Calculate the single average temperature for the WHOLE province
mean_scalar = mean_weather.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=bc_bounds,
    scale=27830 # ERA5 scale is roughly 27.8km
)

# .getInfo() forces Earth Engine to compute it and return a Python dictionary
result = mean_scalar.getInfo()

# Convert Kelvin to Celsius
temp_c = result['temperature_2m'] - 273.15
print(f"Average BC Temperature for July 2019: {temp_c:.2f}°C")

map = geemap.Map(center=[54.0, -125.0], zoom=5)
temp_viz = {
    'min': 273.15,
    'max': 303.15,
    'palette': ['blue', 'purple', 'cyan', 'green', 'yellow', 'red']
}

map.add_basemap('SATELLITE')
map.add_basemap('TERRAIN')

map.addLayer(
    mean_weather.select('temperature_2m'),
    temp_viz,
    'Mean Temperature July 2019'
)

map.addLayer(ee.Image().paint(bc_bounds, 0, 0.5), {}, 'BC Outline Box') #Outline Box

map

Average BC Temperature for July 2019: 13.57°C


Map(center=[54.0, -125.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…